# LLM API Calls

## Setup: API Keys

For the exercises we will be using Groq. Create an API key following the instructions here: [Groq Quickstart](https://console.groq.com/docs/quickstart)

There are many APIs to third-party LLMs, we will use Groq because:
- It provides a free API key, generous free tier, very fast inference (LPU hardware).
- Is OpenAI-compatible-ish Python SDK (`pip install groq`).
- Has good open models available: `llama-3.3-70b-versatile`, `llama-3.1-8b-instant`, `mixtral-8x7b-32768`, `gemma2-9b-it` (exact model list can change — check https://console.groq.com/docs/models).



⚠️ **Never hardcode API keys in your notebooks.** Use environment variables or a `.env` file.

1. Create a file called `.env` in the same folder as this notebook:

```
GROQ_API_KEY=

```
2. Make sure to add your `.env` to your `.gitignore` file so it doesn't get pushed!

3. Then load it with `python-dotenv`.

> **For Ollama (local):** No API key needed! Install Ollama from https://ollama.ai, then run `ollama pull llama3.2` in your terminal. Ollama runs a local server at `http://localhost:11434`.


In [ ]:
# Uncomment and run this to install packages
# !pip install groq langchain-groq python-dotenv

In [1]:


import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # Loads from .env file

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

client_groq = Groq(api_key=GROQ_API_KEY)


## Part 1: Chat Completions API

Modern LLM APIs use a **conversation format** or chat format — a list of messages with roles:

- `system`: Sets the context, persona, or rules for the assistant
- `user`: The human's input
- `assistant`: The model's response (used in multi-turn conversations)

**Note**: The Groq SDK is intentionally identical to the OpenAI SDK — same method names, same response structure, same messages format. This is deliberate: any code written for OpenAI works on Groq with two changes: the `client import` and the `model name`.


In [8]:

response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what a large language model is in 3 sentences."}
    ],
    temperature=0.7,
    max_tokens=300
)
print(response.choices[0].message.content)

A large language model is a type of artificial intelligence (AI) designed to process and understand human language, typically trained on vast amounts of text data to learn patterns and relationships within language. These models use complex algorithms to generate text, answer questions, and even engage in conversation, often with remarkable accuracy and coherence. By leveraging massive datasets and advanced computational power, large language models can learn to recognize, generate, and manipulate language in a way that simulates human-like communication, making them useful for applications such as language translation, text summarization, and chatbots.


### Understanding Generation Parameters

Every model has parameters to control *how* the model samples its next token:

| Parameter | Range | Effect |
|---|---|---|
| `temperature` | 0.0–2.0 | 0 = deterministic; 2 = very random. ~0.7 for most tasks. |
| `top_p` | 0.0–1.0 | Nucleus sampling. 0.9 = only sample from top 90% probability mass. |
| `max_tokens` | 1–context limit | Maximum tokens to generate. Controls response length and cost. |
| `frequency_penalty` | -2.0–2.0 | Penalises repeated tokens. Reduces repetitive outputs. |
| `presence_penalty` | -2.0–2.0 | Penalises tokens that have appeared at all. Encourages topic diversity. |

It's important to be aware of these parameters as they can affect the model response quite a lot!

> 📎 See the cheat sheet: [LLM API Quick Reference](https://amirteymoori.com/llm-parameters-explained-temperature-top-p-top-k/) for a full parameter comparison table.


In [10]:
# Experiment: How does temperature affect output?
prompt = "Write a one-sentence tagline for a new AI startup."

temperatures = [0.0, 0.5, 1.0, 1.5]

print(f"Prompt: '{prompt}'\n")
print(f"{'Temperature':<15} {'Output'}")
print("-" * 80)

for temp in temperatures:
    # Use the steps in the previous cell
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=temp,
        max_tokens=50
    )
    print(f"{temp:<15} {response.choices[0].message.content}")
    

Prompt: 'Write a one-sentence tagline for a new AI startup.'

Temperature     Output
--------------------------------------------------------------------------------
0.0             "Empowering innovation and transforming futures, one intelligent solution at a time, with AI that amplifies human potential."
0.5             "Empowering innovation, amplifying intelligence: where human insight meets artificial genius."
1.0             "Empowering innovation, amplifying intelligence: revolutionizing the future, one connection at a time."
1.5             "Empowering innovation and transforming futures, one intelligent insight at a time, with [Startup Name], the pioneers of Artificial Intelligence solutions."


### ✏️ Exercise 5.1 — Temperature Exploration

Temperature controls how the model responds to a single prompt. But real applications need more than one exchange — they need conversation. The challenge is that the API has no memory: every call starts fresh. The only way to maintain context is to manually pass the entire conversation history with each request. The cell below shows how this works in practice.

Re-run the cell above 3 times with `temperature=0.0`. Does the output change? Now try `temperature=1.5`. What do you notice?

Then answer: for a **customer support chatbot**, which temperature would you choose and why?


In [16]:
# Multi-turn conversation: maintaining context
conversation_history = [
    {"role": "system", "content": "You are a knowledgeable but concise AI tutor."}
]

def chat(user_message: str, history: list) -> str:

    # Step 1: append the user message to history
    history.append({
        "role": "user",
        "content": user_message
    })

    # Step 2: call the API passing the full history
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=history,
        temperature=0.7,
        max_tokens=300
    )

    # Step 3: extract the reply and append to history
    reply = response.choices[0].message.content

    # Add assistant response to history
    history.append({
        "role": "assistant",
        "content": reply
    })

    # Step 4: return the reply
    return reply
    
    # YOUR CODE HERE 
    pass

    

# Simulate a multi-turn conversation
turns = [
    "What is a neural network?",
    "How is that different from a traditional algorithm?",
    "Can you give me a real-world example?"
]

print("Multi-turn conversation demo:")
print("=" * 60)
for turn in turns:
    print(f"\n User: {turn}")
    reply = chat(turn, conversation_history)
    print(f" Assistant: {reply}")

Multi-turn conversation demo:

 User: What is a neural network?
 Assistant: A neural network is a computer system modeled after the human brain. It's composed of layers of interconnected nodes (neurons) that process and transmit information. These nodes apply complex algorithms to recognize patterns, make decisions, and learn from data, allowing the network to perform tasks like image recognition, language translation, and predictive modeling.

 User: How is that different from a traditional algorithm?
 Assistant: A traditional algorithm is a set of instructions that follows a fixed, rule-based approach to solve a problem. It's typically programmed with explicit steps and decision-making rules.

In contrast, a neural network is a non-rule-based system that learns from data and improves over time through a process called training. It doesn't require explicit programming for every possible scenario, instead, it discovers patterns and relationships in the data on its own, making it more f


## Part 2: Running a Model Locally with Ollama

Ollama is a lightweight, open-source framework designed to let you download, run, and manage Large Language Models (LLMs) directly on your own computer. Ollama acts as a user-friendly engine that handles all the complex backend compilation so you can spin up powerful models locally with a single terminal command.

**Why run locally?**
- Zero API cost
- Data never leaves your machine (privacy-sensitive applications)
- Works offline
- Great for development and testing

**However**,  Ollama runs large language models (LLMs) locally on your own hardware. Your hardware capabilities directly dictate the size and speed of the models you can run. The model's weights must fit entirely into your memory to run efficiently.

While Ollama can run on a CPU, it will be incredibly slow (often less than 2–5 tokens per second). A dedicated GPU with high VRAM accelerates processing exponentially. So usually cloud-hosted models are preferred. 



## Minimum System Requirements 
* 8 GB system RAM
* 4-core CPU with AVX2 (Intel Haswell or AMD Zen+)
* 32 GB free storage (SATA SSD acceptable)
* Any GPU with 6 GB+ VRAM or CPU-only
* Linux, macOS, or Windows 10+
Expect: 3B–7B Q4 models at 5–25 tok/s on CPU. Painful but functional.

### Per models
Because Ollama loads AI weights directly into memory (VRAM on dedicated GPUs or unified RAM on Mac/CPU-only systems), match your memory to the parameter size of the model:  
* 1B – 3B Models (e.g., Llama 3.2 1B/3B, Phi-3): 4 GB RAM/VRAM  
* 7B – 8B Models (e.g., Llama 3.1 8B, Mistral 7B, Gemma 2 9B): 
* 8 GB – 12 GB RAM/VRAM  13B – 14B Models (e.g., Qwen 2.5 14B, DeepSeek-R1 14B): 16 GB RAM/VRAM  32B Models (e.g., Qwen 2.5 32B): 
* 32 GB RAM/VRAM70B Models (e.g., Llama 3.3 70B): 64 GB+ RAM/VRAM


**Setup (do this in your terminal before running the cells):**

```bash
# 1. Install Ollama from https://ollama.ai
# 2. Pull a model (3.2 is small and fast)
ollama pull llama3.2
# 3. Ollama runs a server automatically at localhost:11434
```


In [17]:
import requests
import json

OLLAMA_URL = "http://localhost:11434"

def check_ollama():
    """Check if Ollama is running and list available models."""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        models = response.json().get("models", [])
        print(" Ollama is running!")
        print(f"Available models: {[m['name'] for m in models]}")
        return True
    except Exception:
        print(" Ollama is not running. Start it or install from https://ollama.ai")
        return False

check_ollama()

 Ollama is not running. Start it or install from https://ollama.ai


False

In [14]:
def ollama_chat(prompt: str, model: str = "llama3.2", system: str = "") -> str:
    """Call Ollama local API."""
    # Step 1: Build the payload dictionary with:
    # - the model name
    # - messages list with the user prompt
    # - stream set to False
    # - options with temperature 0.7 and num_predict 200
    payload = {
        # FILL IN
    }
    
    # Step 2: If a system prompt is provided, add it to the payload
    # FILL IN
    
    # Step 3: Make a POST request to the Ollama chat endpoint
    # Hint: OLLAMA_URL + "/api/chat"
    try:
        response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=10)
        
        
        # Step 4: Extract and return the message content from the response
        return # FILL IN
        
    except Exception as e:
        return f"Error: {e}"

Compare this to the OpenAI and Anthropic calls. What are the practical differences? When would you choose a local model over a cloud API?


## Part 3 (Paid and optional): Anthropic Claude API

The Anthropic API follows a slightly different structure — notably, the `system` prompt is a separate parameter, not part of the messages list.


In [ ]:
import anthropic

client_ant = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Note: system is a separate parameter in Anthropic's API
# response = client_ant............
response = client_ant.chat.completions.create(
    model="claude-haiku-4-5-20251001",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what a large language model is in 3 sentences."}
    ],
    temperature=0.7,
    max_tokens=200
)


# Step 1: choose a model (use "claude-haiku-4-5-20251001" to keep costs low)
# Step 2: Set max_tokens to 200
# Step 3: Write a system prompt defining the assistant's persona
# Step 4: Add a user message of your choice

print("Model:",
print"Model", 
print("\nResponse:")
print(...................)
print("\nToken usage:")
print(f"  Input:  ...................")
print(f"  Output: ...................")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2988583927.py, line 23)

Compare this code to the OpenAI call. What is the key structural difference? Why do you think Anthropic made `system` a separate parameter rather than part of the messages list?

Have you noticed how ChatGPT and Claude display responses word by word rather than all at once? That's streaming. Without it your app would freeze until the full response is ready — terrible UX for long responses. The **demo** below shows how to implement it.

In [ ]:
# Streaming response — useful for chatbot UX
print("Streaming response demo:")
print("-" * 50)

with client_ant.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "List 5 key skills for an AI engineer in 2025, with one sentence each."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="")

print("\n" + "-" * 50)


## Readings & Sources

- 🌐 [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
- 🌐 [Anthropic API Reference](https://docs.anthropic.com/en/api/getting-started)
- 🌐 [Ollama — run models locally](https://ollama.ai)
- 🌐 [LLM Parameters Guide — amirteymoori.com](https://amirteymoori.com/llm-parameters-explained-temperature-top-p-top-k/) (2025)
- 📘 *Hands-On Large Language Models* — Alammar & Grootendorst (O'Reilly, 2024) — Chapter 3: First Steps with LLM APIs



